# 内存复用

## 1. 功能简介

NPUGraph 本身不支持动态 Shape：当 `dynamic=False` 时，每种 Shape 通常对应一张 FX Graph 和一张 NPUGraph；当 `dynamic=True` 时，多种 Shape 可以复用同一张 FX Graph，但不同 Shape 仍对应不同的 NPUGraph。如果每张 NPUGraph 都申请独立内存，长期运行可能造成显存持续增长，甚至触发 OOM（Out of Memory）。

## 2. 内存复用机制

<table align="left" border="1" cellpadding="6" cellspacing="0">
  <tr><th align="left">Option</th><th align="left">功能描述</th><th align="left">推荐应用场景</th></tr>
  <tr><td align="left"><code>use_graph_pool</code>（模式一）</td><td align="left">传入由 <code>torch.npu.graph_pool_handle()</code> 创建的内存池；默认值为 <code>None</code>。</td><td align="left">让使用同一 options 的多张 NPUGraph 共享指定内存池。</td></tr>
  <tr><td align="left"><code>reuse_graph_pool_in_same_fx</code>（模式二）</td><td align="left">实现同一张 FX Graph 中不同 Shape 对应的多张 NPUGraph 之间的内存复用。默认值为 <code>True</code>。</td><td align="left">适用于 <code>dynamic=True</code> 的多 Shape 推理场景。</td></tr>
  <tr><td align="left"><code>clone_input</code>（模式三）</td><td align="left">对 user_inputs 类输入进行 clone，并复用 clone 后的输入内存。默认值为 <code>True</code>。</td><td align="left">可与模式一或模式二组合；大输入可能带来额外显存压力。</td></tr>
  <tr><td align="left"><code>clone_output</code></td><td align="left">对 NPUGraph 输出进行 clone 后再返回。默认值为 <code>False</code>；输入输出存在别名关系时不会 clone。</td><td align="left">适用于需要跨 Replay 长时间持有输出的场景。</td></tr>
</table>
<div style="clear: both;"></div>

> **注意事项：** 开启模式一时，模式二会自动关闭；模式三可以与模式一或模式二共存。若业务需要跨 Replay 长期持有输出，应开启 `clone_output=True`，并将额外拷贝成本纳入性能与显存评估。

**图 1**  多张 NPUGraph 的内存复用示意图

<img src="./images/graph_memory_reuse.svg" width="850">

多张 NPUGraph 可以通过 Graph Pool 复用显存，但输出内存仍可能在后续 Replay 中被覆盖。需要长期保存输出时，应结合 `clone_output=True` 使用。

In [ ]:
import torch
import torch_npu

assert torch.npu.is_available(), "请在已安装 CANN 和 torch_npu 的昇腾环境中运行"
torch.manual_seed(0)
print("PyTorch:", torch.__version__)
print("torch_npu:", torch_npu.__version__)
print("Device:", torch.npu.get_device_name(0))
# 使用 MatMul + Add 构造包含中间张量的固定 Shape 推理图。
class MatmulAdd(torch.nn.Module):
    def forward(self, x, y, weight):
        return torch.matmul(x, weight) + y


model = MatmulAdd().npu()
# 创建显式图内存池句柄，使同一句柄绑定的 NPUGraph 可以复用池内存。
pool = torch.npu.graph_pool_handle()
compiled = torch.compile(
    model,
    backend="npugraph_ex",
    options={
        "use_graph_pool": pool,
        "clone_input": True,
        "clone_output": True,
    },
    fullgraph=True,
    dynamic=False,
)
x = torch.randn(4, 8, dtype=torch.float16).npu()
y = torch.randn(4, 8, dtype=torch.float16).npu()
w = torch.randn(8, 8, dtype=torch.float16).npu()
# clone_output=True 时，保存的第一次输出不会被后续 Replay 覆盖。
saved = compiled(x, y, w).cpu().clone()
compiled(x, y, w)  # 再次 Replay，验证旧输出仍保持不变
torch.testing.assert_close(compiled(x, y, w).cpu(), saved)
print("已完成 clone_output 结果校验")


In [ ]:
# dynamic=True 场景可启用模式二，复用同一 FX Graph 下多 Shape 的图内存池。
options = {
    "reuse_graph_pool_in_same_fx": True,
    "clone_input": True,
    "clone_output": True,
}

# dynamic=True 时可将 options 传给 torch.compile，随后用多种 shape
# 调用同一张 FX Graph，具体 shape 规则需在目标 NPU 环境确认。
print(options)


## 3. 课后练习

### 一、单选题

（1）【单选题】用于在 options 中显式指定并绑定内存池的选项是？
- A. use_graph_pool
- B. clone_output
- C. force_recapture
- D. pattern_fusion_pass

（2）【单选题】`reuse_graph_pool_in_same_fx` 主要解决哪类场景的复用问题？
- A. 不同进程之间共享 CPU 内存
- B. 同一张 FX Graph 内、多种 Shape 生成的多张 NPUGraph 之间复用内存
- C. 模型参数跨设备复制
- D. 所有输出自动持久化

（3）【单选题】如果业务需要长期保存某一次 Replay 的输出，应开启哪个选项？
- A. force_eager
- B. clone_output
- C. clone_input
- D. dynamic=True

（4）【单选题】动态 Shape 推理中，希望同一 FX Graph 捕获出的多张 NPUGraph 复用内存，应优先考虑？
- A. reuse_graph_pool_in_same_fx
- B. force_recapture
- C. fullgraph=False
- D. input_inplace_pass

（5）【单选题】创建可传给 `use_graph_pool` 的图内存池句柄可使用哪个接口？
- A. torch.npu.graph_pool_handle()
- B. torch.npu.empty_cache()
- C. torch.compile.graph_pool()
- D. torch.npu.clone_output()

（6）【单选题】显式设置 `use_graph_pool` 开启模式一后，模式二会如何处理？
- A. 强制同时开启
- B. 自动关闭
- C. 自动切换到 CPU
- D. 删除已有内存池

### 二、多选题

（7）【多选题】关于内存复用机制的描述，正确的有哪些？
- A. `use_graph_pool` 允许用户显式绑定特定内存池
- B. `reuse_graph_pool_in_same_fx` 面向同一 FX Graph 的多 Shape 场景
- C. `clone_input` 可通过复制 user_inputs 辅助跨图内存复用
- D. 内存复用后不再需要检查输出生命周期

（8）【多选题】验证内存复用策略时，应关注哪些指标？
- A. 多次 Replay 后的显存占用变化
- B. 与 Eager 结果的数值一致性
- C. 稳定区间的端到端耗时
- D. 只看首次编译耗时

（9）【多选题】下列哪些属于多张 NPUGraph 的三种内存复用模式？
- A. use_graph_pool
- B. reuse_graph_pool_in_same_fx
- C. clone_input
- D. force_eager

（10）【多选题】需要长期持有输出时，正确的处理方式有哪些？
- A. 开启 `clone_output=True`
- B. 将额外复制开销纳入显存和性能评估
- C. 在多次 Replay 后检查保存的输出是否保持正确
- D. 假设输出地址永远不会被后续 Replay 复用

**运行以下代码单元查看参考答案与解析。**


In [ ]:
import os
answer_path = "answer/03.04_answer.txt"
if os.path.exists(answer_path):
    with open(answer_path, "r", encoding="utf-8") as f:
        print(f.read())
else:
    print("答案文件未找到，请检查 answer 目录。")
